# 🍽️ Zomato — End-to-End Business Analysis using Python

## Objective
Analyze Zomato's **Customers, Restaurants and Orders** data using:

- **Pandas** — data cleaning, joins, aggregation and business analysis
- **NumPy** — numerical calculations and scoring
- **Matplotlib** — business charts
- **Seaborn** — statistical visualizations

### Business goal
Answer practical questions around:
- Revenue and order performance
- Customer behavior and retention
- Restaurant performance
- City-level performance
- Acquisition channels
- Discounts and delivery fees
- Payment behavior
- Order status / cancellation / refund patterns
- Cuisine performance
- Growth opportunities

**Dataset contains 3 related tables:** `customer`, `restaurants`, `orders`.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Load All Excel Sheets

In [ ]:
FILE_PATH = r"/mnt/data/Zomato_dataset(2).xlsx"

customer = pd.read_excel(FILE_PATH, sheet_name="customer")
restaurants = pd.read_excel(FILE_PATH, sheet_name="restaurants")
orders = pd.read_excel(FILE_PATH, sheet_name="orders")

print("Customer:", customer.shape)
print("Restaurants:", restaurants.shape)
print("Orders:", orders.shape)

display(customer.head())
display(restaurants.head())
display(orders.head())


## 3. Data Dictionary / Structure

In [ ]:
print("CUSTOMER COLUMNS")
print(customer.columns.tolist())

print("\nRESTAURANT COLUMNS")
print(restaurants.columns.tolist())

print("\nORDER COLUMNS")
print(orders.columns.tolist())


## 4. Data Cleaning

In [ ]:
# Standardize column names
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

customer = clean_columns(customer)
restaurants = clean_columns(restaurants)
orders = clean_columns(orders)

# Parse dates
customer["signup_time"] = pd.to_datetime(customer["signup_time"], errors="coerce")
orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"], errors="coerce")

# Numeric columns
for col in ["order_amount", "discount_amount", "delivery_fee"]:
    orders[col] = pd.to_numeric(orders[col], errors="coerce")

# Remove exact duplicates
customer = customer.drop_duplicates().reset_index(drop=True)
restaurants = restaurants.drop_duplicates().reset_index(drop=True)
orders = orders.drop_duplicates().reset_index(drop=True)

print("Duplicates after cleaning:")
print("Customer:", customer.duplicated().sum())
print("Restaurants:", restaurants.duplicated().sum())
print("Orders:", orders.duplicated().sum())


## 5. Data Quality Audit

In [ ]:
def quality_report(df, name):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean()*100).round(2),
        "unique": df.nunique()
    }).sort_values("missing", ascending=False)
    print(f"--- {name} ---")
    display(report)

quality_report(customer, "Customer")
quality_report(restaurants, "Restaurants")
quality_report(orders, "Orders")


## 6. Validate Relationships Between Tables

In [ ]:
print("Customer IDs in orders not found in customer table:",
      orders.loc[~orders["customer_id"].isin(customer["customer_id"]), "customer_id"].nunique())

print("Restaurant IDs in orders not found in restaurant table:",
      orders.loc[~orders["restaurant_id"].isin(restaurants["restaurant_id"]), "restaurant_id"].nunique())

print("Duplicate Customer IDs:", customer["customer_id"].duplicated().sum())
print("Duplicate Restaurant IDs:", restaurants["restaurant_id"].duplicated().sum())
print("Duplicate Order IDs:", orders["order_id"].duplicated().sum())


## 7. Create an Analytical Master Dataset

In [ ]:
master = (
    orders
    .merge(
        customer[["customer_id", "customer_name", "city", "signup_time", "acquisition_channel"]],
        on="customer_id",
        how="left",
        suffixes=("", "_customer")
    )
    .merge(
        restaurants[["restaurant_id", "restaurant_name", "cuisine", "city", "avg_rating"]],
        on="restaurant_id",
        how="left",
        suffixes=("", "_restaurant")
    )
)

# Rename city fields clearly
master = master.rename(columns={
    "city": "customer_city",
    "city_restaurant": "restaurant_city"
})

# Derived metrics
master["gross_order_value"] = master["order_amount"] + master["delivery_fee"]
master["discount_pct"] = np.where(
    master["order_amount"] > 0,
    master["discount_amount"] / master["order_amount"] * 100,
    0
)
master["net_order_value_after_discount"] = (
    master["order_amount"] - master["discount_amount"]
)
master["order_month"] = master["order_timestamp"].dt.to_period("M").astype(str)
master["order_year"] = master["order_timestamp"].dt.year
master["order_day"] = master["order_timestamp"].dt.day_name()

print("Master dataset shape:", master.shape)
display(master.head())


# 📊 Exploratory Data Analysis

## Business Question 1 — What is the overall business performance?
We calculate total orders, delivered orders, revenue, discounts, delivery fees and average order value.


In [ ]:
total_orders = len(master)
delivered = master[master["order_status"].str.lower().eq("delivered")].copy()

kpis = pd.Series({
    "Total Orders": total_orders,
    "Delivered Orders": len(delivered),
    "Total Order Amount": master["order_amount"].sum(),
    "Total Discounts": master["discount_amount"].sum(),
    "Total Delivery Fees": master["delivery_fee"].sum(),
    "Average Order Value": master["order_amount"].mean(),
    "Delivery Success Rate %": len(delivered) / total_orders * 100
}).round(2)

display(kpis.to_frame("Value"))


## Business Question 2 — How are orders distributed by status?

In [ ]:
status_summary = (
    master["order_status"]
    .value_counts()
    .to_frame("Orders")
)
status_summary["Percentage"] = (status_summary["Orders"] / len(master) * 100).round(2)
display(status_summary)

plt.figure(figsize=(9,5))
sns.barplot(x=status_summary.index, y=status_summary["Orders"])
plt.title("Orders by Order Status")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=20)
plt.show()


## Business Question 3 — How has order volume and revenue changed over time?

In [ ]:
monthly = master.groupby("order_month").agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Discounts=("discount_amount", "sum"),
    Delivery_Fees=("delivery_fee", "sum")
).reset_index()

display(monthly.head())

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(monthly["order_month"], monthly["Orders"], marker="o", label="Orders")
ax.set_title("Monthly Order Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Orders")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,6))
sns.lineplot(data=monthly, x="order_month", y="Revenue", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## Business Question 4 — Which cities generate the most orders and revenue?

In [ ]:
city_perf = master.groupby("customer_city").agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Order_Value=("order_amount", "mean"),
    Customers=("customer_id", "nunique")
).sort_values("Revenue", ascending=False)

display(city_perf.head(15))

top_city_plot = city_perf.head(10).sort_values("Revenue")
plt.figure(figsize=(10,6))
sns.barplot(data=top_city_plot.reset_index(), x="Revenue", y="customer_city")
plt.title("Top 10 Cities by Revenue")
plt.xlabel("Revenue")
plt.ylabel("City")
plt.show()


## Business Question 5 — Which restaurants are the top revenue generators?

In [ ]:
restaurant_perf = master.groupby(
    ["restaurant_id", "restaurant_name", "cuisine"]
).agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Order_Value=("order_amount", "mean"),
    Customers=("customer_id", "nunique"),
    Rating=("avg_rating", "first")
).sort_values("Revenue", ascending=False)

display(restaurant_perf.head(15))


## Business Question 6 — Which restaurants have high ratings AND strong customer demand?

In [ ]:
restaurant_perf2 = restaurant_perf.copy()

restaurant_perf2["Demand_Score"] = (
    restaurant_perf2["Orders"].rank(pct=True) * 0.5 +
    restaurant_perf2["Customers"].rank(pct=True) * 0.5
)

restaurant_perf2["Quality_Demand_Score"] = (
    restaurant_perf2["Demand_Score"] * 0.6 +
    (restaurant_perf2["Rating"] / restaurant_perf2["Rating"].max()).fillna(0) * 0.4
)

display(
    restaurant_perf2.sort_values("Quality_Demand_Score", ascending=False).head(15)
)


## Business Question 7 — Which cuisines drive the most revenue?

In [ ]:
cuisine_perf = master.groupby("cuisine").agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Order_Value=("order_amount", "mean"),
    Avg_Rating=("avg_rating", "mean")
).sort_values("Revenue", ascending=False)

display(cuisine_perf)

plt.figure(figsize=(11,6))
plot_data = cuisine_perf.head(10).sort_values("Revenue")
sns.barplot(data=plot_data.reset_index(), x="Revenue", y="cuisine")
plt.title("Top Cuisines by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Cuisine")
plt.show()


## Business Question 8 — Which acquisition channels bring the most customers and revenue?

In [ ]:
acq_perf = master.groupby("acquisition_channel").agg(
    Customers=("customer_id", "nunique"),
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum")
).sort_values("Revenue", ascending=False)

acq_perf["Revenue_per_Customer"] = acq_perf["Revenue"] / acq_perf["Customers"]
acq_perf["Orders_per_Customer"] = acq_perf["Orders"] / acq_perf["Customers"]

display(acq_perf)

plt.figure(figsize=(10,5))
sns.barplot(data=acq_perf.reset_index(), x="Revenue", y="acquisition_channel")
plt.title("Revenue by Customer Acquisition Channel")
plt.xlabel("Revenue")
plt.ylabel("Acquisition Channel")
plt.show()


## Business Question 9 — Which acquisition channels produce high-value customers?

In [ ]:
display(
    acq_perf.sort_values("Revenue_per_Customer", ascending=False)
)


## Business Question 10 — What is the customer repeat-order behavior?

In [ ]:
customer_orders = master.groupby(["customer_id", "customer_name"]).agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Order_Value=("order_amount", "mean")
)

customer_orders["Customer_Type"] = np.where(
    customer_orders["Orders"] == 1,
    "One-time",
    "Repeat"
)

repeat_summary = customer_orders["Customer_Type"].value_counts().to_frame("Customers")
repeat_summary["Percentage"] = (
    repeat_summary["Customers"] / len(customer_orders) * 100
).round(2)

display(repeat_summary)

print("Repeat Customer Rate:",
      round((customer_orders["Orders"] > 1).mean() * 100, 2), "%")


## Business Question 11 — Who are the highest-value customers?

In [ ]:
top_customers = customer_orders.sort_values(
    ["Revenue", "Orders"], ascending=False
).head(20)

display(top_customers)


## Business Question 12 — How much revenue comes from repeat customers?

In [ ]:
customer_type_revenue = customer_orders.groupby("Customer_Type").agg(
    Customers=("Revenue", "count"),
    Revenue=("Revenue", "sum"),
    Avg_Customer_Revenue=("Revenue", "mean")
)

customer_type_revenue["Revenue_Share_%"] = (
    customer_type_revenue["Revenue"] /
    customer_type_revenue["Revenue"].sum() * 100
).round(2)

display(customer_type_revenue)


## Business Question 13 — Are discounts increasing order value or simply reducing realized revenue?

In [ ]:
discount_analysis = master.assign(
    Discount_Band=pd.cut(
        master["discount_pct"],
        bins=[-np.inf, 5, 10, 20, 100, np.inf],
        labels=["0-5%", "5-10%", "10-20%", "20-100%", "100%+"]
    )
)

discount_perf = discount_analysis.groupby("Discount_Band", observed=False).agg(
    Orders=("order_id", "count"),
    Avg_Order_Value=("order_amount", "mean"),
    Avg_Discount=("discount_amount", "mean"),
    Revenue=("order_amount", "sum")
)

display(discount_perf)


## Business Question 14 — What is the relationship between discounts and order amount?

In [ ]:
corr = master[["order_amount", "discount_amount"]].corr()
display(corr)

plt.figure(figsize=(9,6))
sns.scatterplot(
    data=master.sample(min(5000, len(master)), random_state=42),
    x="discount_amount",
    y="order_amount",
    alpha=0.4
)
plt.title("Discount Amount vs Order Amount")
plt.xlabel("Discount Amount")
plt.ylabel("Order Amount")
plt.show()


## Business Question 15 — Which payment modes are most used and which generate the most revenue?

In [ ]:
payment_perf = master.groupby("payment_mode").agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Order_Value=("order_amount", "mean")
).sort_values("Orders", ascending=False)

payment_perf["Order_Share_%"] = (
    payment_perf["Orders"] / payment_perf["Orders"].sum() * 100
).round(2)

display(payment_perf)

plt.figure(figsize=(9,5))
sns.barplot(data=payment_perf.reset_index(), x="Orders", y="payment_mode")
plt.title("Orders by Payment Mode")
plt.xlabel("Orders")
plt.ylabel("Payment Mode")
plt.show()


## Business Question 16 — Which cities have the best restaurant quality?

In [ ]:
city_quality = restaurants.groupby("city").agg(
    Restaurants=("restaurant_id", "count"),
    Avg_Rating=("avg_rating", "mean")
).sort_values("Avg_Rating", ascending=False)

# Avoid tiny samples
city_quality_filtered = city_quality[city_quality["Restaurants"] >= 5]

display(city_quality_filtered.head(15))

plt.figure(figsize=(10,6))
sns.barplot(
    data=city_quality_filtered.head(10).reset_index(),
    x="Avg_Rating",
    y="city"
)
plt.title("Highest Rated Cities — Minimum 5 Restaurants")
plt.xlabel("Average Restaurant Rating")
plt.ylabel("City")
plt.show()


## Business Question 17 — Which restaurant cities have high demand but lower ratings?

In [ ]:
city_opportunity = master.groupby("restaurant_city").agg(
    Orders=("order_id", "count"),
    Revenue=("order_amount", "sum"),
    Avg_Rating=("avg_rating", "mean"),
    Restaurants=("restaurant_id", "nunique")
)

# Opportunity = scale + improvement potential
city_opportunity["Scale_Score"] = city_opportunity["Orders"].rank(pct=True)
city_opportunity["Improvement_Score"] = 1 - city_opportunity["Avg_Rating"].rank(pct=True)

city_opportunity["Opportunity_Score"] = (
    0.6 * city_opportunity["Scale_Score"] +
    0.4 * city_opportunity["Improvement_Score"]
)

display(
    city_opportunity.sort_values("Opportunity_Score", ascending=False).head(15)
)


## Business Question 18 — What days of the week have the strongest order demand?

In [ ]:
day_order = master["order_day"].value_counts()

day_order = day_order.reindex([
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
])

display(day_order.to_frame("Orders"))

plt.figure(figsize=(10,5))
sns.barplot(x=day_order.index, y=day_order.values)
plt.title("Orders by Day of Week")
plt.xlabel("Day")
plt.ylabel("Orders")
plt.xticks(rotation=30)
plt.show()


## Business Question 19 — What is the order value distribution?

In [ ]:
print(master["order_amount"].describe())

plt.figure(figsize=(10,5))
sns.histplot(master["order_amount"], bins=40, kde=True)
plt.title("Order Amount Distribution")
plt.xlabel("Order Amount")
plt.ylabel("Orders")
plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(x=master["order_amount"])
plt.title("Order Amount Boxplot — Outlier Detection")
plt.show()


## Business Question 20 — What are the key correlations among order economics?

In [ ]:
economic_cols = [
    "order_amount",
    "discount_amount",
    "delivery_fee",
    "gross_order_value",
    "net_order_value_after_discount"
]

corr = master[economic_cols].corr()

plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix — Order Economics")
plt.show()

display(corr)


# 🔎 Key Insights Framework

After running the notebook, document the strongest observations using this framework:

### 1. Revenue
- Identify the highest-revenue months, cities, restaurants and cuisines.
- Compare revenue concentration to order concentration.

### 2. Customer
- Measure repeat-customer rate.
- Identify high-value customers.
- Compare acquisition channels by customers, orders and revenue per customer.

### 3. Restaurant
- Find restaurants combining strong rating, demand and revenue.
- Identify high-demand / lower-rating areas for quality improvement.

### 4. Marketing
- Evaluate whether discount-heavy orders generate enough additional value.
- Compare acquisition channels using revenue per customer rather than customer count alone.

### 5. Operations
- Monitor cancelled/refunded orders.
- Identify cities or restaurants where order-status problems are concentrated.

### 6. Pricing
- Examine AOV and discount percentage.
- Do not assume expensive restaurants are automatically better; validate with rating and customer demand.


# 💡 Business Recommendations

## Recommendation 1 — Build a retention program
Target one-time customers with personalized offers based on city, cuisine and previous order value. The objective should be to convert one-time users into repeat customers.

## Recommendation 2 — Optimize acquisition spend
Do not evaluate marketing channels only by number of acquired customers. Compare:
**Revenue per Customer + Orders per Customer + Repeat Rate**.

## Recommendation 3 — Use targeted discounting
Instead of blanket discounts, identify discount bands that produce healthy order values and repeat behavior. Reduce discounts that only reduce revenue without improving customer activity.

## Recommendation 4 — Create restaurant quality interventions
Restaurants with high order volume but lower ratings are important because poor experiences can affect many customers. Prioritize them for quality and service improvement.

## Recommendation 5 — Promote high-value cuisine segments
Use cuisine-level revenue and rating performance to create curated collections and targeted recommendations.

## Recommendation 6 — City-level growth strategy
Prioritize cities with high order/revenue scale and clear quality or restaurant-supply opportunities.

## Recommendation 7 — Strengthen payment and digital adoption
Promote the most convenient payment options while monitoring payment-mode performance and failed/refunded orders.

## Recommendation 8 — Build an executive KPI dashboard
Recommended KPIs:
- Total Orders
- Delivered Orders
- Delivery Success Rate
- Total Revenue
- Average Order Value
- Total Discounts
- Discount %
- Repeat Customer Rate
- Revenue per Customer
- Top Restaurant
- Top City
- Top Cuisine


# 🧠 Final Executive Summary

This project converts a multi-table Zomato dataset into a business-focused analytics solution.

### Analytical flow
**Excel → Data Quality → Table Relationships → Master Dataset → EDA → Business Questions → Visualizations → Insights → Recommendations**

### Portfolio-ready project statement
> Analyzed 50,000+ Zomato orders by integrating customer, restaurant and transaction data. Used Python, Pandas, NumPy, Matplotlib and Seaborn to evaluate revenue, customer retention, acquisition channels, restaurant performance, cuisine trends, discounts, payment behavior and city-level opportunities. Developed business recommendations focused on customer retention, targeted promotions, restaurant quality and geographic growth.

### Skills demonstrated
**Python | Pandas | NumPy | Matplotlib | Seaborn | EDA | Data Cleaning | Data Modeling | Joins | Business Analysis | KPI Analysis | Data Visualization**


## Appendix — Export Analytical Master Dataset (Optional)

In [ ]:
# Uncomment to export the cleaned analytical dataset
# master.to_csv("zomato_master_analysis.csv", index=False)

print("Master analytical dataset is available in the `master` DataFrame.")
